In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [3]:
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer

model_name = "UBC-NLP/MARBERTv2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(
    model_name, num_labels=2
)

sample_tweet = "يا سلام على الجو الجميل اليوم، فعلا روعة!"
inputs = tokenizer(
    sample_tweet, return_tensors="pt", truncation=True, max_length=128
)

with torch.no_grad():
  logits = model(**inputs).logits

print("Token IDs:", inputs["input_ids"])
print("Raw Model Output (Logits):", logits)

config.json:   0%|          | 0.00/757 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/439 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/1.10M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  654MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: UBC-NLP/MARBERTv2
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were ne

Token IDs: tensor([[    2,  2023,  3693,  1977,  4583,  4334,  2221,   112,  3562, 12862,
             5,     3]])
Raw Model Output (Logits): tensor([[0.0276, 0.0902]])


model.safetensors: reconstructing file:   0%|          |  0.00B /  654MB            

model.safetensors: downloading bytes:           |  0.00B            

In [9]:
from datasets import load_dataset

# Load the ArSarcasm-v2 dataset from Hugging Face
dataset = load_dataset("arbml/ArSarcasm_v2")

# Check the available splits (train and test)
print(dataset)

# Print a single sample row from the training split
print("\nSample Training Row:")
print(dataset["train"][0])

README.md:   0%|          | 0.00/563 [00:00<?, ?B/s]

dataset_infos.json:   0%|          | 0.00/3.16k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 1.41MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  342kB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/12548 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/3000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['tweet', 'sentiment', 'dialect', 'label'],
        num_rows: 12548
    })
    test: Dataset({
        features: ['tweet', 'sentiment', 'dialect', 'label'],
        num_rows: 3000
    })
})

Sample Training Row:
{'tweet': '"د. #محمود_العلايلي:أرى أن الفريق #أحمد_شفيق رقم مهم في المعادلة السياسية المصرية ولا يمكن إغفالههل ترى أن هذا صحيح؟#أربعة_زائد_واحد"', 'sentiment': 'NEU', 'dialect': 'msa', 'label': 0}


In [10]:
from transformers import AutoTokenizer

# Load the tokenizer for MARBERTv2
tokenizer = AutoTokenizer.from_pretrained("UBC-NLP/MARBERTv2")


def tokenize_function(examples):
  # Tokenize the tweets with truncation and fixed length
  return tokenizer(
      examples["tweet"], truncation=True, max_length=128, padding="max_length"
  )


# Apply the tokenization across the train and test splits
tokenized_dataset = dataset.map(tokenize_function, batched=True)

print("Tokenized Dataset Ready!")
print(tokenized_dataset)

Map:   0%|          | 0/12548 [00:00<?, ? examples/s]

Map:   0%|          | 0/3000 [00:00<?, ? examples/s]

Tokenized Dataset Ready!
DatasetDict({
    train: Dataset({
        features: ['tweet', 'sentiment', 'dialect', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 12548
    })
    test: Dataset({
        features: ['tweet', 'sentiment', 'dialect', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 3000
    })
})


In [11]:
from transformers import (
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
)

# Load MARBERTv2 configured for 3 sentiment classes (positive, negative, neutral)
model = AutoModelForSequenceClassification.from_pretrained(
    "UBC-NLP/MARBERTv2", num_labels=3
)

# Set up training arguments using the paper's exact hyperparameters
training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=5e-6,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=5,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    fp16=True,  # Cuts VRAM usage in half and speeds up training
)

# Initialize the Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
)

print("Trainer is fully configured and ready!")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: UBC-NLP/MARBERTv2
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were ne

Trainer is fully configured and ready!


In [12]:
# Start the 5-epoch fine-tuning loop
train_result = trainer.train()

# Save the final fine-tuned model and tokenizer locally
trainer.save_model("./marbertv2-arsarcasm-model")
print("Training complete! Model saved.")

/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss
1,No log,0.983545
2,0.791532,0.963895
3,0.539579,1.016278
4,0.457041,1.062833
5,0.457041,1.107414


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


... (3 outputs left out)

/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training complete! Model saved.


In [13]:
import numpy as np
from sklearn.metrics import classification_report, f1_score

# 1. Run predictions on the 3,000 unseen test tweets
predictions_output = trainer.predict(tokenized_dataset["test"])
preds = np.argmax(predictions_output.predictions, axis=-1)
labels = predictions_output.label_ids

# 2. Check the exact label mapping (0, 1, 2)
label_names = dataset["train"].features["label"].names
print("Label Mapping:", list(enumerate(label_names)))

# 3. Print full breakdown (Precision, Recall, F1 per class)
print("\n--- Full Classification Report ---")
print(classification_report(labels, preds, target_names=label_names, digits=4))

# 4. Calculate the paper's primary metric: F_PN (Macro F1 of Positive & Negative only)
pos_idx = label_names.index("positive") if "positive" in label_names else 1
neg_idx = label_names.index("negative") if "negative" in label_names else 0

f1_per_class = f1_score(labels, preds, average=None)
f_pn = (f1_per_class[pos_idx] + f1_per_class[neg_idx]) / 2

print(f"Official Paper Metric (F_PN Score): {f_pn:.4f}")

/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Label Mapping: [(0, 'Non sarcastic'), (1, 'sarcastic')]

--- Full Classification Report ---
               precision    recall  f1-score   support

Non sarcastic     0.8143    0.9114    0.8601      2179
    sarcastic     0.6560    0.4482    0.5326       821

     accuracy                         0.7847      3000
    macro avg     0.7351    0.6798    0.6963      3000
 weighted avg     0.7709    0.7847    0.7705      3000

Official Paper Metric (F_PN Score): 0.6963


In [16]:
from datasets import load_dataset, ClassLabel
import numpy as np
from sklearn.metrics import classification_report, f1_score
from transformers import AutoModelForSequenceClassification, AutoTokenizer, Trainer, TrainingArguments

# 1. Load dataset
dataset = load_dataset("arbml/ArSarcasm_v2")

# Drop the old binary label feature so we can redefine it for 3 classes
dataset = dataset.cast_column("label", ClassLabel(names=["neg", "neu", "pos"]))

label2id = {"neg": 0, "neu": 1, "pos": 2}
id2label = {0: "negative", 1: "neutral", 2: "positive"}


def map_sentiment_labels(example):
  example["label"] = label2id[example["sentiment"].strip().lower()]
  return example


# Map sentiment strings to 0, 1, 2
dataset = dataset.map(map_sentiment_labels)

# 2. Tokenize tweets
model_name = "UBC-NLP/MARBERTv2"
tokenizer = AutoTokenizer.from_pretrained(model_name)


def tokenize_function(examples):
  return tokenizer(
      examples["tweet"], truncation=True, max_length=128, padding="max_length"
  )


tokenized_dataset = dataset.map(tokenize_function, batched=True)

# 3. Load MARBERTv2 for 3-class Sentiment Analysis
model = AutoModelForSequenceClassification.from_pretrained(
    model_name, num_labels=3, id2label=id2label, label2id=label2id
)

# 4. Set up Trainer (using 3 epochs to prevent overfitting)
training_args = TrainingArguments(
    output_dir="./sentiment_results",
    learning_rate=5e-6,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    fp16=True,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
)

# 5. Train and save
trainer.train()
trainer.save_model("./marbertv2-arsarcasm-sentiment")

# 6. Evaluate and calculate the true F_PN score
predictions_output = trainer.predict(tokenized_dataset["test"])
preds = np.argmax(predictions_output.predictions, axis=-1)
labels = predictions_output.label_ids

target_names = ["negative", "neutral", "positive"]
print("\n--- Sentiment Classification Report ---")
print(classification_report(labels, preds, target_names=target_names, digits=4))

# F_PN is the macro F1 of Negative (index 0) and Positive (index 2) only
f1_per_class = f1_score(labels, preds, average=None)
f_pn = (f1_per_class[0] + f1_per_class[2]) / 2
print(f"Official Paper Metric (True F_PN Score): {f_pn:.4f}")

Casting the dataset:   0%|          | 0/12548 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/3000 [00:00<?, ? examples/s]

Map:   0%|          | 0/12548 [00:00<?, ? examples/s]

Map:   0%|          | 0/3000 [00:00<?, ? examples/s]

Map:   0%|          | 0/12548 [00:00<?, ? examples/s]

Map:   0%|          | 0/3000 [00:00<?, ? examples/s]

... (7 outputs left out)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]



--- Sentiment Classification Report ---
              precision    recall  f1-score   support

    negative     0.7936    0.7955    0.7945      1677
     neutral     0.5677    0.4372    0.4940       748
    positive     0.6205    0.8017    0.6995       575

    accuracy                         0.7073      3000
   macro avg     0.6606    0.6781    0.6627      3000
weighted avg     0.7041    0.7073    0.7014      3000

Official Paper Metric (True F_PN Score): 0.7470


In [5]:
from datasets import load_dataset
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
import numpy as np
from sklearn.metrics import classification_report, f1_score
from tqdm import tqdm

# 1. Load test subset
dataset = load_dataset("arbml/ArSarcasm_v2", split="test")
eval_subset = dataset.select(range(500))

# 2. Load Qwen2.5-1.5B-Instruct in FP16
model_id = "Qwen/Qwen2.5-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id, torch_dtype=torch.float16, device_map="auto"
)


# 3. Define zero-shot prompting function with proper token slicing
def predict_sentiment_zeroshot(tweet):
  prompt = f"""Task: Classify the sentiment of the following Arabic tweet into exactly one of these three categories: negative, neutral, or positive.
Tweet: "{tweet}"
Answer with only one word (negative, neutral, or positive):"""

  inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
  input_len = inputs["input_ids"].shape[1]

  outputs = model.generate(
      **inputs, max_new_tokens=10, temperature=0.1, do_sample=False
  )

  # Decode ONLY the newly generated tokens (drops the prompt)
  new_tokens = outputs[0][input_len:]
  response = tokenizer.decode(new_tokens, skip_special_tokens=True).lower()

  if "positive" in response:
    return "pos"
  elif "negative" in response:
    return "neg"
  elif "neutral" in response:
    return "neu"
  else:
    # Fallback if it outputs something else
    return "neu"


# 4. Run zero-shot inference
preds_str = []
true_str = []

print("Running corrected zero-shot generative LLM evaluation...")
for item in tqdm(eval_subset):
  pred = predict_sentiment_zeroshot(item["tweet"])
  preds_str.append(pred)
  true_str.append(item["sentiment"].strip().lower())

# 5. Map strings to integers
str2id = {"neg": 0, "neu": 1, "pos": 2}
preds = [str2id[p] for p in preds_str]
labels = [str2id[t] for t in true_str]

# 6. Print Metrics
target_names = ["negative", "neutral", "positive"]
print("\n--- Corrected Zero-Shot LLM Classification Report ---")
print(classification_report(labels, preds, target_names=target_names, digits=4))

f1_per_class = f1_score(labels, preds, average=None)
f1_neg = f1_per_class[0] if len(f1_per_class) > 0 else 0.0
f1_pos = f1_per_class[2] if len(f1_per_class) > 2 else 0.0
f_pn = (f1_neg + f1_pos) / 2
print(f"Generative LLM Zero-Shot F_PN Score: {f_pn:.4f}")

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Running corrected zero-shot generative LLM evaluation...


100%|██████████| 500/500 [04:41<00:00,  1.78it/s]


--- Corrected Zero-Shot LLM Classification Report ---
              precision    recall  f1-score   support

    negative     0.7078    0.5616    0.6263       276
     neutral     0.3582    0.3556    0.3569       135
    positive     0.3741    0.6180    0.4661        89

    accuracy                         0.5160       500
   macro avg     0.4800    0.5117    0.4831       500
weighted avg     0.5540    0.5160    0.5250       500

Generative LLM Zero-Shot F_PN Score: 0.5462
